# Saved Scenarios Demonstration

This notebook demonstrates PyETM's saved scenario management functionality using the five core runners:

1. **ListSavedScenariosRunner** - List all saved scenarios
2. **CreateSavedScenarioRunner** - Save a scenario to MyETM
3. **FetchSavedScenarioRunner** - Get details of a saved scenario
4. **UpdateSavedScenarioRunner** - Update saved scenario metadata
5. **DeleteSavedScenarioRunner** - Delete a saved scenario

## Setup

In [ ]:
from example_helpers import setup_notebook
setup_notebook()

In [ ]:
from pyetm.clients.base_client import BaseClient
from pyetm.models.scenario import Scenario
from pyetm.services.scenario_runners.list_saved_scenarios import ListSavedScenariosRunner
from pyetm.services.scenario_runners.create_saved_scenario import CreateSavedScenarioRunner
from pyetm.services.scenario_runners.fetch_saved_scenario import FetchSavedScenarioRunner
from pyetm.services.scenario_runners.update_saved_scenario import UpdateSavedScenarioRunner
from pyetm.services.scenario_runners.delete_saved_scenario import DeleteSavedScenarioRunner

client = BaseClient()

## 1. List Saved Scenarios

List existing saved scenarios to see what's already available.

In [ ]:
result = ListSavedScenariosRunner.run(client, page=1, limit=10)

if result.success:
    print(f"Found {len(result.data)} saved scenarios:")
    for saved in result.data[:5]:
        print(f"  [{saved['id']}] {saved['title']}")
else:
    print(f"Error: {result.errors}")

## 2. Create a Saved Scenario

First create a session scenario, then save it to MyETM.

In [ ]:
# Create a session scenario
scenario = Scenario.new(area_code="nl", end_year=2050)
print(f"Created session scenario {scenario.id}")

# Save it to MyETM
result = CreateSavedScenarioRunner.run(
    client,
    saved_scenario_data={
        "scenario_id": scenario.id,
        "title": "PyETM Demo Scenario",
        "description": "A demonstration of saved scenario runners",
        "private": False
    }
)

if result.success:
    saved_id = result.data['id']
    print(f"Created saved scenario {saved_id}")
else:
    print(f"Error: {result.errors}")

## 3. Fetch Saved Scenario Details

Retrieve detailed information about the saved scenario.

In [ ]:
if result.success:
    # Create a simple object with an id attribute
    class SavedScenarioRef:
        def __init__(self, saved_id):
            self.id = saved_id

    saved_ref = SavedScenarioRef(saved_id)
    fetch_result = FetchSavedScenarioRunner.run(client, saved_ref)

    if fetch_result.success:
        data = fetch_result.data
        print(f"Saved Scenario {data['id']}:")
        print(f"  Title: {data['title']}")
        print(f"  Description: {data.get('description')}")
        print(f"  Scenario ID: {data['scenario_id']}")
        print(f"  Private: {data['private']}")
    else:
        print(f"Error: {fetch_result.errors}")

## 4. Update Saved Scenario Metadata

Update the title, description, or privacy setting.

In [ ]:
if result.success:
    saved_ref = SavedScenarioRef(saved_id)
    update_result = UpdateSavedScenarioRunner.run(
        client,
        saved_ref,
        update_data={
            "title": "PyETM Demo Scenario (Updated)",
            "description": "Updated description to demonstrate the update functionality"
        }
    )

    if update_result.success:
        print("Updated successfully:")
        print(f"  New title: {update_result.data['title']}")
        print(f"  New description: {update_result.data.get('description')}")
    else:
        print(f"Error: {update_result.errors}")

## 5. Delete Saved Scenario

Clean up by deleting the saved scenario. Note: This only deletes the saved scenario record, not the underlying session scenario.

In [ ]:
if result.success:
    saved_ref = SavedScenarioRef(saved_id)
    delete_result = DeleteSavedScenarioRunner.run(client, saved_ref)

    if delete_result.success:
        print(f"Deleted saved scenario {saved_id}")
    else:
        print(f"Error: {delete_result.errors}")

## Verify Deletion

List saved scenarios again to confirm deletion.

In [ ]:
if result.success:
    verify_result = ListSavedScenariosRunner.run(client, page=1, limit=10)

    if verify_result.success:
        found = any(s['id'] == saved_id for s in verify_result.data)
        if found:
            print(f"Saved scenario {saved_id} still exists")
        else:
            print(f"Saved scenario {saved_id} successfully deleted")
    else:
        print(f"Error: {verify_result.errors}")